
# System-One-подобная модель: обучение в Colab

Небольшой открытый аналог класса моделей "System One" (см. TypeSafe AI / Jev):

- **Не авторегрессивная** — модель не генерирует текст, а параллельно скорит заранее заданные варианты ответа на заранее заданные вопросы (schema-based decisions).
- **Мультимодальная** — вход: текстовое состояние (`state_text`) **+ опциональная картинка**.
- **Калиброванная** — обучается strictly proper scoring rule (soft cross-entropy), плюс post-hoc temperature scaling и ECE-метрика.
- **"Не может ошибиться в типах"** — ответ всегда один из заранее перечисленных вариантов (closed-set softmax), а не свободный текст.

### Архитектура (bi-encoder scorer)

```
state_text ──┐
             ├─► text_encoder ──► state_repr  ┐
question ────┘                                 │
                                                ├─► context_mlp ──► context vector
image (опц.)─► vision_encoder ──► image_repr  ┘

option_1 ─► text_encoder ─► option_repr_1 ─┐
option_2 ─► text_encoder ─► option_repr_2 ─┼─► score_i = <context, option_repr_i> / sqrt(D)
...                                        ─┘
                                              softmax по вариантам одного вопроса ─► калиброванная вероятность
```

Это позволяет:
- поддерживать **произвольное число вопросов и вариантов ответа** на пример (в отличие от фиксированных classification heads);
- переиспользовать один и тот же текстовый энкодер для state/question/options — не нужно обучать отдельную голову под каждое поле схемы;
- считать все варианты **одним batched forward pass** (параллельно, не авторегрессивно).

⚠️ Это учебная реконструкция по публичному блог-посту TypeSafe (пейпера нет), а не воспроизведение их реальной архитектуры. Цель — рабочий, дешёвый, полностью открытый прототип того же *класса* моделей.

**Рекомендуемая среда:** Colab, GPU (Runtime → Change runtime type → T4/L4). Обучение маленьких энкодеров (~150-400M параметров) с LoRA/полным fine-tuning занимает от десятков минут до пары часов на T4.


In [ ]:
#@title Установка зависимостей
!pip install -q transformers accelerate datasets pillow scikit-learn matplotlib safetensors sentencepiece
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
#@title Конфиг
import os

CFG = dict(
    # Текстовый энкодер: длинный контекст, компактный, открытый.
    text_model_name = "answerdotai/ModernBERT-base",   # 150M, ctx=8192
    # Визуальный энкодер: маленький CLIP vision tower.
    vision_model_name = "openai/clip-vit-base-patch32", # ~88M
    d_model = 512,
    max_options = 16,          # максимум вариантов ответа на вопрос (паддинг/маска для остальных)
    max_state_len = 1024,      # токенов на state_text
    max_question_len = 64,
    max_option_len = 32,
    batch_size = 8,
    lr = 2e-5,
    head_lr = 1e-3,            # для новых слоёв (проекции, MLP) — учим быстрее, чем сам энкодер
    epochs = 3,
    freeze_vision = True,      # заморозить CLIP vision tower целиком (дешевле, обычно достаточно)
    freeze_text_layers = 0,    # сколько нижних слоёв текстового энкодера заморозить (0 = не морозить)
    train_path = "train.jsonl",
    val_path = "val.jsonl",
    out_dir = "checkpoints",
    seed = 42,
)

os.makedirs(CFG["out_dir"], exist_ok=True)
import random, numpy as np
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])



## Формат датасета

Один пример — одна строка JSONL. Датасет генерируется твоим LLM API (см. текстовый гайд в чате) и должен выглядеть так:

```json
{
  "id": "ex_001",
  "state_text": "Тикет от клиента: 'Приложение крашится каждый раз при открытии отчёта, уже 3 дня, я плачу за Pro-план'. История: 2 предыдущих тикета за месяц, план Pro, LTV $840.",
  "image_path": null,
  "questions": [
    {
      "key": "churn_risk",
      "text": "Насколько высок риск оттока этого клиента?",
      "options": ["низкий", "средний", "высокий"],
      "target_probs": [0.05, 0.25, 0.70]
    },
    {
      "key": "priority",
      "text": "Какой приоритет должен получить тикет?",
      "options": ["low", "normal", "high", "urgent"],
      "target_probs": [0.0, 0.0, 0.15, 0.85]
    }
  ]
}
```

Правила:
- `image_path` — путь к файлу картинки (относительно папки датасета) или `null`, если картинки нет. Модель поддерживает оба случая в одном датасете.
- `target_probs` — **мягкое распределение** (сумма = 1), полученное от учителя (например, самоконсистентностью большой LLM). Если у тебя только точный ответ без вероятностей — просто поставь `1.0` на правильный вариант и `0.0` на остальные (hard label), это тоже валидно (частный случай).
- Число вариантов ответа на вопрос — любое, но не больше `CFG["max_options"]` (лишние будут обрезаны с предупреждением).
- Разные примеры могут иметь разное число вопросов и разные схемы — модель это не фиксирует жёстко.


In [ ]:
#@title (Опционально) Сгенерировать игрушечный датасет, чтобы ноутбук работал "из коробки"
# Это ТОЛЬКО для проверки пайплайна. Замени train.jsonl / val.jsonl на датасет,
# сгенерированный твоей LLM (см. гайд по генерации данных).
import json, random

TOY_EXAMPLES = []
templates = [
    ("Клиент жалуется на баг в приложении уже несколько дней, платит за Pro-план.",
     "churn_risk", "Риск оттока клиента?", ["низкий", "средний", "высокий"], [0.05, 0.25, 0.70]),
    ("Клиент похвалил новую фичу и попросил добавить экспорт в PDF.",
     "churn_risk", "Риск оттока клиента?", ["низкий", "средний", "высокий"], [0.85, 0.12, 0.03]),
    ("Заказ на маленькую сумму, доставлен вовремя, отзывов нет.",
     "priority", "Приоритет обработки тикета?", ["low", "normal", "high", "urgent"], [0.6, 0.3, 0.08, 0.02]),
    ("Крупный корпоративный клиент сообщает о падении продакшена.",
     "priority", "Приоритет обработки тикета?", ["low", "normal", "high", "urgent"], [0.0, 0.02, 0.28, 0.70]),
]

random.seed(0)
for i in range(200):
    state, key, qtext, options, probs = random.choice(templates)
    # немного шума, чтобы target_probs не были идентичными
    noisy = [max(0.001, p + random.uniform(-0.05, 0.05)) for p in probs]
    s = sum(noisy)
    noisy = [p / s for p in noisy]
    TOY_EXAMPLES.append({
        "id": f"toy_{i}",
        "state_text": state,
        "image_path": None,
        "questions": [{"key": key, "text": qtext, "options": options, "target_probs": noisy}],
    })

split = int(0.85 * len(TOY_EXAMPLES))
with open(CFG["train_path"], "w") as f:
    for ex in TOY_EXAMPLES[:split]:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
with open(CFG["val_path"], "w") as f:
    for ex in TOY_EXAMPLES[split:]:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"train: {split}, val: {len(TOY_EXAMPLES) - split}")
print("Пример:", TOY_EXAMPLES[0])


In [ ]:
#@title Dataset и collate_fn
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoImageProcessor

tokenizer = AutoTokenizer.from_pretrained(CFG["text_model_name"])
image_processor = AutoImageProcessor.from_pretrained(CFG["vision_model_name"])


class SchemaDataset(Dataset):
    # Разворачивает каждый (example, question) в отдельный обучающий сэмпл.

    def __init__(self, path):
        self.rows = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                ex = json.loads(line)
                for q in ex["questions"]:
                    self.rows.append({
                        "state_text": ex["state_text"],
                        "image_path": ex.get("image_path"),
                        "question_text": q["text"],
                        "options": q["options"][: CFG["max_options"]],
                        "target_probs": q["target_probs"][: CFG["max_options"]],
                        "key": q.get("key", ""),
                    })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def _tok(texts, max_len):
    return tokenizer(
        texts, padding="max_length", truncation=True,
        max_length=max_len, return_tensors="pt",
    )


def collate_fn(batch):
    B = len(batch)
    MAXOPT = CFG["max_options"]

    state_enc = _tok([b["state_text"] for b in batch], CFG["max_state_len"])
    question_enc = _tok([b["question_text"] for b in batch], CFG["max_question_len"])

    # опции: паддим до MAXOPT штук текстом "" (потом замаскируем)
    flat_option_texts = []
    option_valid_mask = torch.zeros(B, MAXOPT, dtype=torch.long)
    target_probs = torch.zeros(B, MAXOPT, dtype=torch.float)

    for i, b in enumerate(batch):
        n = len(b["options"])
        s = sum(b["target_probs"]) or 1.0
        for j in range(MAXOPT):
            if j < n:
                flat_option_texts.append(b["options"][j])
                option_valid_mask[i, j] = 1
                target_probs[i, j] = b["target_probs"][j] / s
            else:
                flat_option_texts.append(tokenizer.pad_token or "")

    option_enc = _tok(flat_option_texts, CFG["max_option_len"])
    option_ids = option_enc["input_ids"].view(B, MAXOPT, -1)
    option_mask = option_enc["attention_mask"].view(B, MAXOPT, -1)

    has_image = torch.tensor([1 if b["image_path"] else 0 for b in batch], dtype=torch.long)
    if has_image.sum() > 0:
        imgs = []
        for b in batch:
            if b["image_path"]:
                img = Image.open(b["image_path"]).convert("RGB")
            else:
                img = Image.new("RGB", (224, 224))  # заглушка, замаскируется has_image=0
            imgs.append(img)
        pixel_values = image_processor(images=imgs, return_tensors="pt")["pixel_values"]
    else:
        pixel_values = None

    return {
        "state_ids": state_enc["input_ids"], "state_mask": state_enc["attention_mask"],
        "question_ids": question_enc["input_ids"], "question_mask": question_enc["attention_mask"],
        "option_ids": option_ids, "option_mask": option_mask,
        "option_valid_mask": option_valid_mask,
        "target_probs": target_probs,
        "pixel_values": pixel_values, "has_image": has_image,
    }


train_ds = SchemaDataset(CFG["train_path"])
val_ds = SchemaDataset(CFG["val_path"])
train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate_fn)
val_dl = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn)
print(f"train samples (по вопросам): {len(train_ds)}, val: {len(val_ds)}")


In [ ]:
#@title Модель: мультимодальный bi-encoder scorer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, CLIPVisionModelWithProjection


class MultimodalSchemaScorer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.text_enc = AutoModel.from_pretrained(cfg["text_model_name"])
        self.vision_enc = CLIPVisionModelWithProjection.from_pretrained(cfg["vision_model_name"])

        if cfg["freeze_vision"]:
            for p in self.vision_enc.parameters():
                p.requires_grad = False

        if cfg["freeze_text_layers"] > 0:
            # заморозить нижние N слоёв энкодера (экономит память/время, часто без потери качества)
            layers = self.text_enc.encoder.layers if hasattr(self.text_enc, "encoder") else []
            for layer in layers[: cfg["freeze_text_layers"]]:
                for p in layer.parameters():
                    p.requires_grad = False

        text_dim = self.text_enc.config.hidden_size
        vision_dim = self.vision_enc.config.projection_dim
        d = cfg["d_model"]

        self.state_proj = nn.Linear(text_dim, d)
        self.question_proj = nn.Linear(text_dim, d)
        self.option_proj = nn.Linear(text_dim, d)
        self.image_proj = nn.Linear(vision_dim, d)
        self.null_image = nn.Parameter(torch.zeros(d))

        self.context_mlp = nn.Sequential(
            nn.Linear(d * 3, d * 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(d * 2, d)
        )
        self.d_model = d

    def _pool(self, input_ids, attention_mask):
        out = self.text_enc(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state                       # [N, L, H]
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)  # mean pooling
        return pooled

    def forward(self, state_ids, state_mask, question_ids, question_mask,
                option_ids, option_mask, option_valid_mask,
                pixel_values=None, has_image=None):
        B, MAXOPT, Lo = option_ids.shape

        state_repr = self.state_proj(self._pool(state_ids, state_mask))
        question_repr = self.question_proj(self._pool(question_ids, question_mask))

        if pixel_values is not None:
            vis_out = self.vision_enc(pixel_values=pixel_values).image_embeds  # [B, vision_dim]
            img_repr = self.image_proj(vis_out)
            has_image_f = has_image.float().unsqueeze(-1)
            img_repr = img_repr * has_image_f + self.null_image.unsqueeze(0) * (1 - has_image_f)
        else:
            img_repr = self.null_image.unsqueeze(0).expand(B, -1)

        context = self.context_mlp(torch.cat([state_repr, question_repr, img_repr], dim=-1))  # [B, d]

        flat_ids = option_ids.reshape(B * MAXOPT, Lo)
        flat_mask = option_mask.reshape(B * MAXOPT, Lo)
        opt_repr = self.option_proj(self._pool(flat_ids, flat_mask)).view(B, MAXOPT, self.d_model)

        scores = torch.einsum("bd,bod->bo", context, opt_repr) / (self.d_model ** 0.5)
        scores = scores.masked_fill(option_valid_mask == 0, float("-inf"))
        return scores


def soft_ce_loss(scores, target_probs):
    logp = F.log_softmax(scores, dim=-1)
    logp = torch.where(torch.isfinite(logp), logp, torch.zeros_like(logp))
    return -(target_probs * logp).sum(dim=-1).mean()


In [ ]:
#@title Тренировочный цикл
import torch
from torch.optim import AdamW

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultimodalSchemaScorer(CFG).to(device)

new_params, backbone_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (new_params if any(k in name for k in ["proj", "context_mlp", "null_image"]) else backbone_params).append(p)

optimizer = AdamW([
    {"params": backbone_params, "lr": CFG["lr"]},
    {"params": new_params, "lr": CFG["head_lr"]},
])


def move_batch(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def run_epoch(dl, train=True):
    model.train(train)
    total_loss, n = 0.0, 0
    for batch in dl:
        batch = move_batch(batch, device)
        with torch.set_grad_enabled(train):
            scores = model(
                batch["state_ids"], batch["state_mask"],
                batch["question_ids"], batch["question_mask"],
                batch["option_ids"], batch["option_mask"], batch["option_valid_mask"],
                batch["pixel_values"], batch["has_image"],
            )
            loss = soft_ce_loss(scores, batch["target_probs"])
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * batch["state_ids"].size(0)
        n += batch["state_ids"].size(0)
    return total_loss / max(n, 1)


for epoch in range(CFG["epochs"]):
    train_loss = run_epoch(train_dl, train=True)
    val_loss = run_epoch(val_dl, train=False)
    print(f"epoch {epoch+1}/{CFG['epochs']}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

torch.save(model.state_dict(), f"{CFG['out_dir']}/model.pt")
print("Сохранено:", f"{CFG['out_dir']}/model.pt")



## Калибровка (temperature scaling) и ECE

Cross-entropy сама по себе уже даёт разумно калиброванные вероятности (strictly proper scoring rule), но нейросети обычно немного overconfident. Стандартное лекарство — **temperature scaling**: один скалярный параметр `T`, на который делятся логиты перед softmax, подбирается на val-сете *после* обучения (веса модели не трогаем).


In [ ]:
#@title Temperature scaling + ECE
import torch
import torch.nn as nn

@torch.no_grad()
def collect_val_scores(model, dl, device):
    all_scores, all_targets, all_valid = [], [], []
    model.eval()
    for batch in dl:
        batch = move_batch(batch, device)
        scores = model(
            batch["state_ids"], batch["state_mask"],
            batch["question_ids"], batch["question_mask"],
            batch["option_ids"], batch["option_mask"], batch["option_valid_mask"],
            batch["pixel_values"], batch["has_image"],
        )
        all_scores.append(scores.cpu())
        all_targets.append(batch["target_probs"].cpu())
        all_valid.append(batch["option_valid_mask"].cpu())
    return torch.cat(all_scores), torch.cat(all_targets), torch.cat(all_valid)


val_scores, val_targets, val_valid = collect_val_scores(model, val_dl, device)

log_T = nn.Parameter(torch.zeros(1))  # T = exp(log_T), стартуем с T=1
opt_T = torch.optim.LBFGS([log_T], lr=0.05, max_iter=100)

def closure():
    opt_T.zero_grad()
    T = log_T.exp()
    loss = soft_ce_loss(val_scores / T, val_targets)
    loss.backward()
    return loss

opt_T.step(closure)
T_optimal = log_T.exp().item()
print(f"Оптимальная температура T = {T_optimal:.3f}")


def expected_calibration_error(scores, targets, valid_mask, n_bins=10):
    # ECE по топ-1 предсказанию каждого вопроса.
    probs = torch.softmax(scores.masked_fill(valid_mask == 0, float("-inf")), dim=-1)
    conf, pred = probs.max(dim=-1)
    true_idx = targets.argmax(dim=-1)
    correct = (pred == true_idx).float()

    bins = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        acc = correct[mask].mean()
        avg_conf = conf[mask].mean()
        ece += (mask.float().mean()) * (avg_conf - acc).abs()
    return ece.item()


ece_before = expected_calibration_error(val_scores, val_targets, val_valid)
ece_after = expected_calibration_error(val_scores / T_optimal, val_targets, val_valid)
print(f"ECE до калибровки:  {ece_before:.4f}")
print(f"ECE после калибровки: {ece_after:.4f}")

import json
with open(f"{CFG['out_dir']}/calibration.json", "w") as f:
    json.dump({"temperature": T_optimal}, f)


In [ ]:
#@title Reliability diagram
import matplotlib.pyplot as plt
import torch

def reliability_diagram(scores, targets, valid_mask, n_bins=10, title="Reliability diagram"):
    probs = torch.softmax(scores.masked_fill(valid_mask == 0, float("-inf")), dim=-1)
    conf, pred = probs.max(dim=-1)
    true_idx = targets.argmax(dim=-1)
    correct = (pred == true_idx).float()

    bins = torch.linspace(0, 1, n_bins + 1)
    accs, confs, counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            accs.append(0); confs.append((lo + hi) / 2 * 100); counts.append(0)
            continue
        accs.append(correct[mask].mean().item())
        confs.append(conf[mask].mean().item())
        counts.append(mask.sum().item())

    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "k--", label="идеальная калибровка")
    plt.bar([b.item() for b in bins[:-1]], accs, width=1 / n_bins, align="edge",
            alpha=0.6, edgecolor="black", label="реальная точность")
    plt.xlabel("Уверенность модели")
    plt.ylabel("Точность")
    plt.title(title)
    plt.legend()
    plt.show()

reliability_diagram(val_scores, val_targets, val_valid, title="До калибровки (T=1)")
reliability_diagram(val_scores / T_optimal, val_targets, val_valid, title=f"После калибровки (T={T_optimal:.2f})")



## Инференс

Функция ниже — это по сути и есть "вызов Jev": даёшь текстовое состояние (+опционально картинку) и список вопросов со схемой ответов, получаешь калиброванные вероятности по каждому варианту.


In [ ]:
#@title Инференс
import torch
from PIL import Image

@torch.no_grad()
def predict(state_text, questions, image_path=None, temperature=None):
    # questions: [{"key": str, "text": str, "options": [str, ...]}]
    # Возвращает: [{"key":..., "prediction":..., "confidence":..., "probs": {opt: p}}]
    model.eval()
    temperature = temperature or T_optimal
    results = []

    for q in questions:
        batch = collate_fn([{
            "state_text": state_text,
            "image_path": image_path,
            "question_text": q["text"],
            "options": q["options"][: CFG["max_options"]],
            "target_probs": [0.0] * len(q["options"][: CFG["max_options"]]),  # не используется на инференсе
            "key": q.get("key", ""),
        }])
        batch = move_batch(batch, device)
        scores = model(
            batch["state_ids"], batch["state_mask"],
            batch["question_ids"], batch["question_mask"],
            batch["option_ids"], batch["option_mask"], batch["option_valid_mask"],
            batch["pixel_values"], batch["has_image"],
        )
        probs = torch.softmax(scores[0] / temperature, dim=-1)
        n_opt = len(q["options"])
        probs = probs[:n_opt].cpu()
        best = probs.argmax().item()
        results.append({
            "key": q.get("key", ""),
            "prediction": q["options"][best],
            "confidence": probs[best].item(),
            "probs": {opt: round(p, 4) for opt, p in zip(q["options"], probs.tolist())},
        })
    return results


# Пример вызова:
example_result = predict(
    state_text="Клиент пишет: приложение крашится третий день подряд, план Pro, уже писал раньше.",
    questions=[
        {"key": "churn_risk", "text": "Риск оттока клиента?", "options": ["низкий", "средний", "высокий"]},
        {"key": "priority", "text": "Приоритет обработки тикета?", "options": ["low", "normal", "high", "urgent"]},
    ],
    image_path=None,
)
for r in example_result:
    print(r)



## Экспорт и ускорение (для прод-инференса)

- **Квантизация**: `torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` даёт 2-4x на CPU почти без потери качества.
- **ONNX**: экспортируй `text_enc` и `vision_enc` отдельно через `torch.onnx.export`, потом собери граф в ONNX Runtime — обычно даёт наибольший speedup для энкодеров такого размера.
- **Кэш state/question embedding**: если один и тот же `state_text` используется для многих вопросов подряд, посчитай `state_repr` один раз и переиспользуй — вопросы/картинка меняются, state — нет.
- **Кэш option embeddings**: если набор вариантов ответа фиксирован (например, всегда 4 уровня приоритета), их эмбеддинги можно посчитать один раз офлайн и не пересчитывать на каждый запрос — тогда на инференс остаётся только encode(state)+encode(question)+MLP+dot-product, что очень дёшево.
- Следующий шаг масштабирования: заменить `ModernBERT-base` (150M) на `ModernBERT-large` (395M) или добавить LoRA вместо полного fine-tuning для более крупных бэкбонов при том же бюджете GPU-памяти.


In [ ]:
#@title (Опционально) Скачать чекпоинт себе на диск
from google.colab import files
# files.download(f"{CFG['out_dir']}/model.pt")
# files.download(f"{CFG['out_dir']}/calibration.json")
print("Раскомментируй строки выше, чтобы скачать чекпоинт и калибровку локально.")
